# Signal Engine — Intraday Forecasting with Risk Management

**How to use:**
1. Edit **Cell 2 (CONFIG)** only — set your asset, timeframe, and account size.
2. Run all cells top-to-bottom (`Kernel → Restart & Run All`).
3. The signal cell prints entry / TP / SL and position size.
4. Optionally enable the live scanner (last cell) for continuous updates.

**Prediction engine:** AutoTheta + AutoETS ensemble (statsforecast).  
**Risk management:** ATR-based SL, fixed-fractional position sizing, R:R filter.  
**Extra filters:** ADX trend strength, RSI regime, multi-TF bias alignment.

In [ ]:
import sys, os, warnings, threading, time
warnings.filterwarnings('ignore')
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timezone

from statsforecast import StatsForecast
from statsforecast.models import AutoTheta, AutoETS

from src.data_feeds          import fetch_multi_timeframe, SYMBOL_MAP
from src.feature_engineering import compute_features, build_multi_tf_snapshot, atr as calc_atr, rsi as calc_rsi
from src.signal_engine       import _score_timeframe
from src.signal_logger       import SignalLogger

plt.rcParams.update({
    'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False
})

print('Imports OK')
print('Supported assets:', list(SYMBOL_MAP.keys()))

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              EDIT ONLY THIS CELL                           ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Asset & Timeframe ──────────────────────────────────────────
ASSET    = 'XAUUSD'   # XAUUSD | EURUSD | GBPUSD | USDJPY | USDCAD | GBPJPY
ENTRY_TF = '15min'    # 1min | 5min | 15min | 30min | 1H | 4H | 1D

# ── Prediction ─────────────────────────────────────────────────
TRAIN_BARS  = 200     # bars of history fed to AutoTheta/AutoETS
FORECAST_H  = 3       # how many bars ahead to predict

# ── Risk Management ────────────────────────────────────────────
ACCOUNT_BALANCE = 1000.0   # your account size in USD
MAX_RISK_PCT    = 0.01     # max risk per trade  (1% of account)
RR_RATIO        = 2.0      # reward-to-risk ratio
ATR_SL_MULT     = 1.5      # SL = entry +/- ATR_SL_MULT x ATR(14)
MIN_RR          = 1.5      # skip signal if R:R < this

# ── Filters ────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 55  # min confidence % to issue a signal
ADX_TREND_THRESHOLD  = 20  # skip when ADX < this (choppy market)
RSI_OVERBOUGHT       = 75  # skip BUY when RSI > this
RSI_OVERSOLD         = 25  # skip SELL when RSI < this

# ── TF stack for multi-timeframe bias ──────────────────────────
_TF_STACKS = {
    '1min':  ['1min',  '5min',  '15min'],
    '5min':  ['5min',  '15min', '1H'],
    '15min': ['15min', '1H',    '4H'],
    '30min': ['30min', '1H',    '4H'],
    '1H':    ['1H',    '4H',    '1D'],
    '4H':    ['4H',    '1D'],
    '1D':    ['1D'],
}
TIMEFRAMES = _TF_STACKS.get(ENTRY_TF, [ENTRY_TF])

print(f'Asset      : {ASSET}')
print(f'Entry TF   : {ENTRY_TF}  |  TF stack: {TIMEFRAMES}')
print(f'Train bars : {TRAIN_BARS}  |  Forecast: {FORECAST_H} bars ahead')
print(f'Account    : ${ACCOUNT_BALANCE:,.0f}  |  Risk/trade: {MAX_RISK_PCT:.1%}')
print(f'R:R target : 1:{RR_RATIO}  |  ATR SL mult: {ATR_SL_MULT}x  |  Min R:R: {MIN_RR}')
print(f'Filters    : Conf>={CONFIDENCE_THRESHOLD}%  ADX>={ADX_TREND_THRESHOLD}  RSI OB={RSI_OVERBOUGHT} OS={RSI_OVERSOLD}')

In [ ]:
print(f'Fetching {ASSET} data ...')
tf_data = fetch_multi_timeframe(ASSET, TIMEFRAMES, bars=500)

for tf, df in tf_data.items():
    atr14 = calc_atr(df, 14).iloc[-1]
    print(f'  {tf:6s}: {len(df):4d} bars | last={df["close"].iloc[-1]:.3f} | ATR(14)={atr14:.3f} | {df.index[-1]}')

entry_df  = tf_data[ENTRY_TF]
cur_price = float(entry_df['close'].iloc[-1])
cur_atr   = float(calc_atr(entry_df, 14).iloc[-1])
cur_time  = entry_df.index[-1]

print(f'\nCurrent price : {cur_price:.5f}')
print(f'Current ATR14 : {cur_atr:.5f}')
print(f'Last bar time : {cur_time}')

In [ ]:
# ── AutoTheta + AutoETS ensemble forecast ────────────────────────────────

def _season_len(tf):
    return {'1min': 390, '5min': 78, '15min': 26, '30min': 13,
            '1H': 7, '4H': 2, '1D': 5}.get(tf, 10)

def run_statsforecast(df, tf, train_bars, h):
    close   = df['close'].dropna()
    log_ret = np.log(close / close.shift(1)).dropna()
    series  = log_ret.iloc[-train_bars:].values.astype(float)
    n, sl   = len(series), _season_len(tf)

    sf_df = pd.DataFrame({
        'unique_id': ['asset'] * n,
        'ds': pd.date_range('2000-01-01', periods=n, freq='15min'),
        'y': series,
    })
    preds = StatsForecast(
        models=[AutoTheta(season_length=sl), AutoETS(season_length=sl)],
        freq='15min', n_jobs=1
    ).forecast(df=sf_df, h=h)

    theta = preds['AutoTheta'].values
    ets   = preds['AutoETS'].values
    ens   = (theta + ets) / 2
    last_px = float(close.iloc[-1])
    cum     = np.cumsum(ens)

    return {
        'theta':       theta,
        'ets':         ets,
        'ensemble':    ens,
        'pred_prices': [last_px * np.exp(cum[i]) for i in range(h)],
        'direction':   int(np.sign(ens[0])),
        'pred_return': float(ens[0]),
    }

print('Running AutoTheta + AutoETS ...')
fc = run_statsforecast(entry_df, ENTRY_TF, TRAIN_BARS, FORECAST_H)

dir_label = 'BUY' if fc['direction'] > 0 else ('SELL' if fc['direction'] < 0 else 'NEUTRAL')
print(f'\nModel forecast ({ENTRY_TF}, {FORECAST_H} bars ahead):')
print(f'  Direction   : {dir_label}')
print(f'  Pred return : {fc["pred_return"]:+.5f}  ({fc["pred_return"]*100:+.3f}%)')
print(f'  Pred prices : {", ".join([f"{p:.3f}" for p in fc["pred_prices"]])}')
print()
print(f'  {"Bar":5s}  {"AutoTheta":>12s}  {"AutoETS":>12s}  {"Ensemble":>12s}')
for i, (t, e, ens) in enumerate(zip(fc['theta'], fc['ets'], fc['ensemble'])):
    print(f'  Bar+{i+1}  {t:>+12.5f}  {e:>+12.5f}  {ens:>+12.5f}')

In [ ]:
# ── Market context: multi-TF bias + all indicators ────────────────────────

snapshots = build_multi_tf_snapshot(tf_data)
scores    = {tf: _score_timeframe(snap) for tf, snap in snapshots.items()}
buy_tfs   = [tf for tf, sc in scores.items() if sc['direction'] ==  1]
sell_tfs  = [tf for tf, sc in scores.items() if sc['direction'] == -1]

print('Multi-timeframe bias:')
print(f'  {"TF":6s}  {"Direction":10s}  {"Strength":8s}  Factors')
print('  ' + '-'*70)
for tf, sc in scores.items():
    d = {1: 'BULLISH', -1: 'BEARISH', 0: 'NEUTRAL'}[sc['direction']]
    print(f'  {tf:6s}  {d:10s}  {sc["strength"]:.3f}     {", ".join(sc["factors"][:5])}')

# ── All indicators via compute_features (single source of truth) ──────────
feat     = compute_features(entry_df)
rsi_v    = float(feat['rsi_14'].iloc[-1])   if 'rsi_14'    in feat.columns else 50.0
adx_v    = float(feat['adx_14'].iloc[-1])   if 'adx_14'    in feat.columns else 0.0
plus_di  = float(feat['plus_di'].iloc[-1])  if 'plus_di'   in feat.columns else 0.0
minus_di = float(feat['minus_di'].iloc[-1]) if 'minus_di'  in feat.columns else 0.0
ema20    = float(feat['ema_20'].iloc[-1])   if 'ema_20'    in feat.columns else cur_price
ema50    = float(feat['ema_50'].iloc[-1])   if 'ema_50'    in feat.columns else cur_price
macd_h   = float(feat['macd_hist'].iloc[-1])if 'macd_hist' in feat.columns else 0.0
stoch_k  = float(feat['stoch_k'].iloc[-1])  if 'stoch_k'   in feat.columns else 50.0
bb_pct   = float(feat['bb_pct'].iloc[-1])   if 'bb_pct'    in feat.columns else 0.5

# Volatility regime: current ATR vs 20-bar ATR average
atr_ma     = calc_atr(entry_df, 14).rolling(20).mean().iloc[-1]
vol_regime = 'HIGH' if cur_atr > atr_ma * 1.2 else ('LOW' if cur_atr < atr_ma * 0.8 else 'NORMAL')

print(f'\nEntry-TF indicators ({ENTRY_TF}):')
print(f'  RSI(14)    : {rsi_v:.1f}  {"[OVERBOUGHT]" if rsi_v > RSI_OVERBOUGHT else "[OVERSOLD]" if rsi_v < RSI_OVERSOLD else ""}')
print(f'  Stoch %K   : {stoch_k:.1f}')
print(f'  ADX(14)    : {adx_v:.1f}  +DI={plus_di:.1f}  -DI={minus_di:.1f}  {"[TRENDING]" if adx_v >= ADX_TREND_THRESHOLD else "[RANGING]"}') 
print(f'  MACD hist  : {macd_h:+.5f}  {"BULLISH" if macd_h > 0 else "BEARISH"}')
print(f'  EMA20/50   : {ema20:.3f} / {ema50:.3f}  Price {"ABOVE" if cur_price > ema20 else "BELOW"} EMA20')
print(f'  BB %B      : {bb_pct:.2f}  (0=lower band, 0.5=mid, 1=upper band)')
print(f'  ATR regime : {vol_regime}  (cur={cur_atr:.4f}  avg={atr_ma:.4f})')

In [ ]:
# ── Signal: combine model + filters → entry / TP / SL ────────────────────

def compute_signal():
    model_dir = fc['direction']   # +1 BUY / -1 SELL / 0 neutral
    if model_dir == 0:
        return None, 'Model is NEUTRAL'

    signal = 'BUY' if model_dir > 0 else 'SELL'

    # RSI extreme filter
    if signal == 'BUY'  and rsi_v > RSI_OVERBOUGHT:
        return None, f'RSI={rsi_v:.0f} overbought — skip BUY'
    if signal == 'SELL' and rsi_v < RSI_OVERSOLD:
        return None, f'RSI={rsi_v:.0f} oversold — skip SELL'

    aligned_tfs  = buy_tfs  if signal == 'BUY' else sell_tfs
    opposite_tfs = sell_tfs if signal == 'BUY' else buy_tfs
    total_tfs    = len(scores)
    align_pct    = len(aligned_tfs) / total_tfs * 100 if total_tfs > 0 else 0

    # ── Confidence score (0-95) ───────────────────────────────────────────
    conf = 0
    conf += min(35, int(align_pct * 0.5))                          # TF alignment
    conf += 20 if adx_v >= ADX_TREND_THRESHOLD else 5             # ADX trending
    conf += 15 if abs(fc['pred_return']) > 0.0003 else 8          # model magnitude
    conf += 15 if ((macd_h > 0) == (signal == 'BUY')) else 0      # MACD agrees
    conf += 10 if ((cur_price > ema20) == (signal == 'BUY')) else 0  # price vs EMA20
    conf += 5  if len(opposite_tfs) == 0 else 0                    # no opposing TFs
    conf  = min(conf, 95)

    if conf < CONFIDENCE_THRESHOLD:
        return None, f'Confidence={conf}% < threshold {CONFIDENCE_THRESHOLD}%'

    # ── Entry / SL / TP ───────────────────────────────────────────────────
    entry   = cur_price
    sl_dist = cur_atr * ATR_SL_MULT
    tp_dist = sl_dist * RR_RATIO

    sl = entry - sl_dist if signal == 'BUY' else entry + sl_dist
    tp = entry + tp_dist if signal == 'BUY' else entry - tp_dist

    tp_ladder = [p for p in fc['pred_prices']
                 if (p > entry if signal == 'BUY' else p < entry)]

    if tp_dist / sl_dist < MIN_RR:
        return None, f'R:R={tp_dist/sl_dist:.2f} < minimum {MIN_RR}'

    return {
        'signal': signal, 'entry': entry, 'sl': sl, 'tp': tp,
        'tp_ladder': tp_ladder, 'sl_dist': sl_dist, 'tp_dist': tp_dist,
        'rr': tp_dist / sl_dist, 'confidence': conf,
        'adx': adx_v, 'rsi': rsi_v, 'vol_regime': vol_regime,
        'aligned_tfs': aligned_tfs, 'pred_return': fc['pred_return'],
    }, None

sig, reason = compute_signal()

print('=' * 55)
if sig:
    arrow = '^' if sig['signal'] == 'BUY' else 'v'
    print(f'  SIGNAL  : {sig["signal"]} {arrow}  {ASSET}  [{ENTRY_TF}]')
    print(f'  Time    : {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}')
    print(f'  Conf    : {sig["confidence"]}%')
    print('-' * 55)
    print(f'  Entry      : {sig["entry"]:.5f}')
    print(f'  Stop Loss  : {sig["sl"]:.5f}  ({sig["sl_dist"]:.5f} pts  = {ATR_SL_MULT}x ATR)')
    print(f'  Take Profit: {sig["tp"]:.5f}  ({sig["tp_dist"]:.5f} pts)')
    print(f'  R:R        : 1:{sig["rr"]:.2f}')
    if sig['tp_ladder']:
        print(f'  TP ladder  : {", ".join([f"{p:.3f}" for p in sig["tp_ladder"]])}')
    print('-' * 55)
    print(f'  ADX(14)   : {sig["adx"]:.1f}  |  RSI(14): {sig["rsi"]:.1f}')
    print(f'  Vol regime: {sig["vol_regime"]}  |  Model pred: {sig["pred_return"]:+.4f}')
    print(f'  Aligned TFs: {sig["aligned_tfs"]}')
else:
    print(f'  NO SIGNAL')
    print(f'  Reason  : {reason}')
print('=' * 55)

In [ ]:
# ── Position sizing & full trade plan ────────────────────────────────────

if sig is None:
    print('No signal — skip position sizing.')
else:
    risk_usd = ACCOUNT_BALANCE * MAX_RISK_PCT

    # Volatility scaling: reduce 25% in high-vol
    vol_scale  = 0.75 if sig['vol_regime'] == 'HIGH' else 1.0

    # Confidence scaling
    if   sig['confidence'] >= 75: conf_scale = 1.00
    elif sig['confidence'] >= 60: conf_scale = 0.75
    else:                         conf_scale = 0.50

    adj_risk  = risk_usd * vol_scale * conf_scale
    sl_points = abs(sig['entry'] - sig['sl'])
    is_gold   = ASSET in ('XAUUSD', 'XAGUSD')

    if is_gold:
        position_units = adj_risk / sl_points
        unit_label     = 'oz'
        reward_usd     = position_units * abs(sig['tp'] - sig['entry'])
    else:
        pip_size      = 0.0001 if 'JPY' not in ASSET else 0.01
        sl_pips       = sl_points / pip_size
        position_units = round(adj_risk / (sl_pips * 10.0), 2)
        unit_label    = 'lots'
        reward_usd    = position_units * (abs(sig['tp'] - sig['entry']) / pip_size) * 10.0

    print('RISK MANAGEMENT')
    print('=' * 55)
    print(f'  Account balance  : ${ACCOUNT_BALANCE:,.2f}')
    print(f'  Base risk        : ${risk_usd:.2f}  ({MAX_RISK_PCT:.1%})')
    print(f'  Volatility scale : {vol_scale:.0%}  ({sig["vol_regime"]} vol)')
    print(f'  Confidence scale : {conf_scale:.0%}  ({sig["confidence"]}% conf)')
    print(f'  Adjusted risk    : ${adj_risk:.2f}')
    print('-' * 55)
    print(f'  SL distance      : {sl_points:.5f}')
    print(f'  Position size    : {position_units:.4f} {unit_label}')
    print(f'  Max loss (SL hit): ${adj_risk:.2f}  ({adj_risk/ACCOUNT_BALANCE:.2%} of account)')
    print(f'  Max gain (TP hit): ${reward_usd:.2f}  ({reward_usd/ACCOUNT_BALANCE:.2%} of account)')
    print(f'  R:R              : 1:{reward_usd/adj_risk:.2f}')
    print('=' * 55)
    print()
    print('TRADE PLAN')
    print(f'  {sig["signal"]} {ASSET} @ {sig["entry"]:.5f}')
    print(f'  Size : {position_units:.4f} {unit_label}')
    print(f'  SL   : {sig["sl"]:.5f}  (risk ${adj_risk:.2f})')
    print(f'  TP   : {sig["tp"]:.5f}  (reward ${reward_usd:.2f})')

In [ ]:
# ── Chart: candlesticks + EMA + RSI + MACD + signal levels ───────────────

LOOKBACK  = 80
df_plot   = entry_df.tail(LOOKBACK).copy()
feat_plot = compute_features(entry_df).tail(LOOKBACK)
n, xs     = len(df_plot), range(len(df_plot))

fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1, figsize=(16, 11), gridspec_kw={'height_ratios': [3, 1, 1]}, sharex=True
)

# ── Candlesticks ──────────────────────────────────────────────────────────
for i, (_, row) in enumerate(df_plot.iterrows()):
    col = 'forestgreen' if row['close'] >= row['open'] else 'tomato'
    ax1.plot([i, i], [row['low'], row['high']], color=col, lw=0.8)
    ax1.bar(i, abs(row['close'] - row['open']),
            bottom=min(row['open'], row['close']), color=col, alpha=0.85, width=0.6)

# EMAs
for col_name, color, label in [('ema_20', 'gold', 'EMA20'), ('ema_50', 'skyblue', 'EMA50')]:
    if col_name in feat_plot.columns:
        ax1.plot(xs, feat_plot[col_name].values, color=color, lw=1.3, label=label, zorder=3)

# Signal levels
if sig:
    ax1.axhline(sig['entry'], color='royalblue',   ls='--', lw=2.0, label=f'Entry {sig["entry"]:.3f}')
    ax1.axhline(sig['tp'],    color='forestgreen',  ls='-',  lw=2.0, label=f'TP {sig["tp"]:.3f}')
    ax1.axhline(sig['sl'],    color='tomato',        ls='-',  lw=2.0, label=f'SL {sig["sl"]:.3f}')
    ax1.axhspan(min(sig['entry'], sig['tp']), max(sig['entry'], sig['tp']), alpha=0.07, color='forestgreen')
    ax1.axhspan(min(sig['entry'], sig['sl']), max(sig['entry'], sig['sl']), alpha=0.07, color='tomato')
    for lp in sig.get('tp_ladder', []):
        ax1.axhline(lp, color='limegreen', ls=':', lw=1.0, alpha=0.6)

title = (f'{sig["signal"]} | Conf={sig["confidence"]}% | R:R=1:{sig["rr"]:.1f}'
         if sig else 'NO SIGNAL')
ax1.set_title(f'{ASSET} {ENTRY_TF}  |  {title}  |  '
              f'{datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}',
              fontweight='bold', fontsize=11)
ax1.set_ylabel('Price')
ax1.legend(fontsize=7, loc='upper left', ncol=5)

# ── RSI panel ─────────────────────────────────────────────────────────────
rsi_vals = calc_rsi(df_plot['close'], 14).values
ax2.plot(xs, rsi_vals, color='mediumpurple', lw=1.3)
ax2.axhline(RSI_OVERBOUGHT, color='tomato',      ls='--', lw=0.8)
ax2.axhline(RSI_OVERSOLD,   color='forestgreen', ls='--', lw=0.8)
ax2.axhline(50,             color='gray',        ls=':',  lw=0.6)
ax2.fill_between(xs, rsi_vals, RSI_OVERBOUGHT, where=(rsi_vals > RSI_OVERBOUGHT), alpha=0.2, color='tomato')
ax2.fill_between(xs, rsi_vals, RSI_OVERSOLD,   where=(rsi_vals < RSI_OVERSOLD),   alpha=0.2, color='forestgreen')
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI(14)', fontsize=8)

# ── MACD histogram panel ───────────────────────────────────────────────────
if 'macd_hist' in feat_plot.columns:
    mh = feat_plot['macd_hist'].values
    ax3.bar(xs, mh, color=['forestgreen' if v >= 0 else 'tomato' for v in mh], alpha=0.75)
    ax3.axhline(0, color='black', lw=0.8)
ax3.set_ylabel('MACD hist', fontsize=8)

# X labels
ticks = list(range(0, n, max(1, n // 8)))
ax3.set_xticks(ticks)
ax3.set_xticklabels(
    [df_plot.index[t].strftime('%m-%d %H:%M') for t in ticks],
    rotation=30, ha='right', fontsize=7
)

plt.tight_layout()
out_path = '../results/new_models/plots/signal_chart.png'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Chart saved: {out_path}')

In [ ]:
# ── Live scanner ─────────────────────────────────────────────────────────
# Set LIVE=True to start, LIVE=False and re-run this cell to stop.

LIVE            = False
SCAN_INTERVAL_S = 60    # seconds between scans

if 'live_stop' not in dir():
    live_stop = threading.Event()

def _live_worker():
    _count = 0
    _last  = None
    while not live_stop.is_set():
        _count += 1
        now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
        try:
            fresh   = fetch_multi_timeframe(ASSET, TIMEFRAMES, bars=500)
            f_entry = fresh[ENTRY_TF]
            f_price = float(f_entry['close'].iloc[-1])
            f_atr   = float(calc_atr(f_entry, 14).iloc[-1])
            f_fc    = run_statsforecast(f_entry, ENTRY_TF, TRAIN_BARS, FORECAST_H)
            bar_id  = str(f_entry.index[-1])
            dir_lbl = 'BUY' if f_fc['direction'] > 0 else ('SELL' if f_fc['direction'] < 0 else 'FLAT')

            if dir_lbl != 'FLAT' and bar_id != _last:
                _last = bar_id
                sl_d  = f_atr * ATR_SL_MULT
                tp_d  = sl_d  * RR_RATIO
                e     = f_price
                sl    = e - sl_d if dir_lbl == 'BUY' else e + sl_d
                tp    = e + tp_d if dir_lbl == 'BUY' else e - tp_d
                print(f'[{now}][{_count:4d}] *** {dir_lbl} {ASSET}  '
                      f'entry={e:.3f}  SL={sl:.3f}  TP={tp:.3f}  '
                      f'pred={f_fc["pred_return"]:+.4f} ***')
            else:
                print(f'[{now}][{_count:4d}] {dir_lbl:4s} | {f_price:.3f} | pred={f_fc["pred_return"]:+.5f}')
        except Exception as exc:
            print(f'[{now}] Error: {exc}')
        live_stop.wait(SCAN_INTERVAL_S)
    print('Scanner stopped.')

if LIVE:
    live_stop.clear()
    threading.Thread(target=_live_worker, daemon=True).start()
    print(f'Live scanner ON: {ASSET} {ENTRY_TF}, every {SCAN_INTERVAL_S}s')
    print('Set LIVE=False and re-run this cell to stop.')
else:
    live_stop.set()
    print('Scanner OFF.  Set LIVE=True and re-run to start.')